In [0]:
import requests
import json
import os
import datetime
import time
from pyspark.sql import SparkSession
from pyspark.sql.types import *

# Base volume paths for raw JSON storage
EPA_VOLUME = "/Workspace/Users/ksyogueboule@gmail.com/nearme/volume/EPA/"
NOAA_VOLUME = "/Workspace/Users/ksyogueboule@gmail.com/nearme/volume/NOAA/"
USGS_VOLUME = "/Workspace/Users/ksyogueboule@gmail.com/nearme/volume/USGS/"
NCDOT_VOLUME = "/Workspace/Users/ksyogueboule@gmail.com/nearme/volume/NCDOT/"

# Create directories if they don't exist
for vol in [EPA_VOLUME, NOAA_VOLUME, USGS_VOLUME, NCDOT_VOLUME]:
    os.makedirs(vol, exist_ok=True)
    print(f"Ensured directory exists: {vol}")

# Timestamp for this run
run_ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"Run timestamp: {run_ts}")
print("Setup complete.")

In [0]:
# Cell 1 — EPA Air Quality Data (AirNow API)
# Note: AirNow web service was retired Oct 1, 2026 (see https://docs.airnowapi.org/webservices/retired)
# The .env key is loaded for when a replacement API becomes available.
# Primary source is now Open-Meteo Air Quality API (free, no key).

# Load AIRNOW_API_KEY from .env file
env_path = "/Workspace/Users/ksyogueboule@gmail.com/nearme/.env"
API_KEY = ""
if os.path.exists(env_path):
    with open(env_path) as f:
        for line in f:
            if line.strip().startswith("AIRNOW_API_KEY="):
                API_KEY = line.strip().split("=", 1)[1]
                break
    print(f"Loaded AIRNOW_API_KEY from .env file")
else:
    print(f"WARNING: .env file not found at {env_path}")
epa_data = None
epa_source = None

# Attempt 1: AirNow API (retired Oct 1, 2026 — kept for when replacement launches)
if API_KEY:
    try:
        airnow_url = f"https://www.airnowapi.org/aq/observation/zipCode/current/?format=application/json&zipCode=27606&distance=25&API_KEY={API_KEY}"
        resp = requests.get(airnow_url, timeout=15)
        resp.raise_for_status()
        epa_data = resp.json()
        epa_source = "AirNow API"
        print(f"Fetched from AirNow API: {len(epa_data)} observations")
    except Exception as e:
        print(f"AirNow API failed: {e}")

# Attempt 2: AirNow public endpoint (no key)
if epa_data is None:
    try:
        public_url = "https://www.airnow.gov/aqf/observation/zipCode/current/?format=application/json&zipCode=27606&distance=25"
        resp = requests.get(public_url, timeout=15, headers={"User-Agent": "nearme-hackathon/1.0"})
        resp.raise_for_status()
        epa_data = resp.json()
        epa_source = "AirNow Public"
        print(f"Fetched from AirNow public endpoint: {len(epa_data)} observations")
    except Exception as e:
        print(f"AirNow public endpoint failed: {e}")

# Attempt 3: Open-Meteo Air Quality API (free, no key — robust fallback)
if epa_data is None:
    try:
        openmeteo_url = (
            "https://air-quality-api.open-meteo.com/v1/air-quality"
            "?latitude=35.78&longitude=-78.64"
            "&current=us_aqi,pm10,pm2_5,ozone,nitrogen_dioxide,sulphur_dioxide,carbon_monoxide"
            "&timezone=America/New_York"
        )
        resp = requests.get(openmeteo_url, timeout=15)
        resp.raise_for_status()
        epa_data = resp.json()
        epa_source = "Open-Meteo Air Quality"
        print(f"Fetched from Open-Meteo Air Quality API")
    except Exception as e:
        print(f"Open-Meteo Air Quality API failed: {e}")

# Save raw JSON
epa_file = os.path.join(EPA_VOLUME, f"epa_air_quality_{run_ts}.json")
if epa_data is not None:
    with open(epa_file, "w") as f:
        json.dump(epa_data, f, indent=2)
    print(f"Saved raw JSON to {epa_file}")
    print(f"Data source: {epa_source}")
else:
    print("WARNING: All EPA data sources failed. No data saved.")
    epa_data = {"error": "All EPA data sources failed", "timestamp": run_ts}
    with open(epa_file, "w") as f:
        json.dump(epa_data, f, indent=2)

# Parse into Spark DataFrame
if epa_source == "Open-Meteo Air Quality":
    current = epa_data.get("current", {})
    epa_rows = [{
        "source": epa_source,
        "latitude": 35.78,
        "longitude": -78.64,
        "us_aqi": current.get("us_aqi"),
        "pm10": current.get("pm10"),
        "pm2_5": current.get("pm2_5"),
        "o3": current.get("ozone"),
        "no2": current.get("nitrogen_dioxide"),
        "so2": current.get("sulphur_dioxide"),
        "co": current.get("carbon_monoxide"),
        "measurement_time": current.get("time"),
        "interval": current.get("interval"),
        "ingested_at": run_ts,
    }]
elif isinstance(epa_data, list) and len(epa_data) > 0:
    epa_rows = []
    for obs in epa_data:
        epa_rows.append({
            "source": epa_source,
            "latitude": obs.get("Latitude"),
            "longitude": obs.get("Longitude"),
            "us_aqi": obs.get("AQI"),
            "category": obs.get("Category", {}).get("Name") if isinstance(obs.get("Category"), dict) else obs.get("Category"),
            "parameter": obs.get("ParameterName"),
            "unit": obs.get("Unit"),
            "value": obs.get("Value"),
            "measurement_time": obs.get("DateObserved") + " " + obs.get("HourObserved", "") + ":00" if obs.get("DateObserved") else None,
            "ingested_at": run_ts,
        })
else:
    epa_rows = [{
        "source": "N/A",
        "latitude": None,
        "longitude": None,
        "us_aqi": None,
        "pm10": None,
        "pm2_5": None,
        "o3": None,
        "no2": None,
        "so2": None,
        "co": None,
        "measurement_time": None,
        "interval": None,
        "ingested_at": run_ts,
    }]

epa_schema = StructType([
    StructField("source", StringType()),
    StructField("latitude", DoubleType()),
    StructField("longitude", DoubleType()),
    StructField("us_aqi", DoubleType()),
    StructField("category", StringType()),
    StructField("parameter", StringType()),
    StructField("unit", StringType()),
    StructField("value", DoubleType()),
    StructField("pm10", DoubleType()),
    StructField("pm2_5", DoubleType()),
    StructField("o3", DoubleType()),
    StructField("no2", DoubleType()),
    StructField("so2", DoubleType()),
    StructField("co", DoubleType()),
    StructField("measurement_time", StringType()),
    StructField("interval", StringType()),
    StructField("ingested_at", StringType()),
])

epa_df = spark.createDataFrame(epa_rows, schema=epa_schema)
print("\nEPA Air Quality DataFrame:")
display(epa_df)

In [0]:
# Cell 2 — NOAA Weather Data (NWS API)
# NWS API is free, no API key required

headers = {"User-Agent": "nearme-hackathon/1.0 (contact@example.com)"}

# 1. Fetch active weather alerts for North Carolina
try:
    alerts_url = "https://api.weather.gov/alerts?area=NC"
    resp = requests.get(alerts_url, headers=headers, timeout=15)
    resp.raise_for_status()
    noaa_alerts = resp.json()
    print(f"Fetched {len(noaa_alerts.get('features', []))} weather alerts for NC")
except Exception as e:
    print(f"Failed to fetch NOAA alerts: {e}")
    noaa_alerts = {"features": []}

# 2. Fetch observation stations for NC
try:
    stations_url = "https://api.weather.gov/stations?state=NC"
    resp = requests.get(stations_url, headers=headers, timeout=15)
    resp.raise_for_status()
    noaa_stations = resp.json()
    station_list = noaa_stations.get("features", [])
    print(f"Fetched {len(station_list)} observation stations for NC")
except Exception as e:
    print(f"Failed to fetch NOAA stations: {e}")
    noaa_stations = {"features": []}
    station_list = []

# 3. Fetch latest observations for first 10 stations
observations = []
stations_to_query = station_list[:10]
print(f"\nFetching latest observations for {len(stations_to_query)} stations...")
for station in stations_to_query:
    station_id = station.get("properties", {}).get("stationIdentifier")
    station_name = station.get("properties", {}).get("name", "Unknown")
    if not station_id:
        continue
    try:
        obs_url = f"https://api.weather.gov/stations/{station_id}/observations/latest"
        resp = requests.get(obs_url, headers=headers, timeout=15)
        resp.raise_for_status()
        obs = resp.json()
        obs_props = obs.get("properties", {})
        observations.append({
            "station_id": station_id,
            "station_name": station_name,
            "temperature": obs_props.get("temperature", {}).get("value") if isinstance(obs_props.get("temperature"), dict) else None,
            "temperature_unit": obs_props.get("temperature", {}).get("unitCode") if isinstance(obs_props.get("temperature"), dict) else None,
            "wind_speed": obs_props.get("windSpeed", {}).get("value") if isinstance(obs_props.get("windSpeed"), dict) else None,
            "wind_speed_unit": obs_props.get("windSpeed", {}).get("unitCode") if isinstance(obs_props.get("windSpeed"), dict) else None,
            "humidity": obs_props.get("relativeHumidity", {}).get("value") if isinstance(obs_props.get("relativeHumidity"), dict) else None,
            "barometric_pressure": obs_props.get("barometricPressure", {}).get("value") if isinstance(obs_props.get("barometricPressure"), dict) else None,
            "pressure_unit": obs_props.get("barometricPressure", {}).get("unitCode") if isinstance(obs_props.get("barometricPressure"), dict) else None,
            "wind_direction": obs_props.get("windDirection", {}).get("value") if isinstance(obs_props.get("windDirection"), dict) else None,
            "description": obs_props.get("textDescription"),
            "observation_time": obs_props.get("timestamp"),
            "ingested_at": run_ts,
        })
        print(f"  {station_id} ({station_name}): OK")
    except Exception as e:
        print(f"  {station_id} ({station_name}): {e}")
    time.sleep(0.5)  # Rate limiting

print(f"\nCollected {len(observations)} observations")

# 4. Save raw JSON files
alerts_file = os.path.join(NOAA_VOLUME, f"noaa_alerts_{run_ts}.json")
with open(alerts_file, "w") as f:
    json.dump(noaa_alerts, f, indent=2)
print(f"Saved alerts to {alerts_file}")

stations_file = os.path.join(NOAA_VOLUME, f"noaa_stations_{run_ts}.json")
with open(stations_file, "w") as f:
    json.dump(noaa_stations, f, indent=2)
print(f"Saved stations to {stations_file}")

observations_file = os.path.join(NOAA_VOLUME, f"noaa_observations_{run_ts}.json")
with open(observations_file, "w") as f:
    json.dump(observations, f, indent=2)
print(f"Saved observations to {observations_file}")

# 5. Parse observations into Spark DataFrame
if len(observations) == 0:
    observations = [{
        "station_id": None, "station_name": None, "temperature": None,
        "temperature_unit": None, "wind_speed": None, "wind_speed_unit": None,
        "humidity": None, "barometric_pressure": None, "pressure_unit": None,
        "wind_direction": None, "description": None, "observation_time": None,
        "ingested_at": run_ts,
    }]

noaa_obs_schema = StructType([
    StructField("station_id", StringType()),
    StructField("station_name", StringType()),
    StructField("temperature", DoubleType()),
    StructField("temperature_unit", StringType()),
    StructField("wind_speed", DoubleType()),
    StructField("wind_speed_unit", StringType()),
    StructField("humidity", DoubleType()),
    StructField("barometric_pressure", DoubleType()),
    StructField("pressure_unit", StringType()),
    StructField("wind_direction", DoubleType()),
    StructField("description", StringType()),
    StructField("observation_time", StringType()),
    StructField("ingested_at", StringType()),
])
noaa_obs_df = spark.createDataFrame(observations, schema=noaa_obs_schema)
print("\nNOAA Weather Observations DataFrame:")
display(noaa_obs_df)

# Also parse alerts into a DataFrame
alert_rows = []
for feature in noaa_alerts.get("features", []):
    props = feature.get("properties", {})
    alert_rows.append({
        "alert_id": props.get("id"),
        "event": props.get("event"),
        "headline": props.get("headline"),
        "description": props.get("description"),
        "severity": props.get("severity"),
        "certainty": props.get("certainty"),
        "urgency": props.get("urgency"),
        "area_desc": props.get("areaDesc"),
        "effective": props.get("effective"),
        "expires": props.get("expires"),
        "ingested_at": run_ts,
    })

if len(alert_rows) == 0:
    alert_rows = [{
        "alert_id": None, "event": None, "headline": None, "description": None,
        "severity": None, "certainty": None, "urgency": None, "area_desc": None,
        "effective": None, "expires": None, "ingested_at": run_ts,
    }]

noaa_alerts_schema = StructType([
    StructField("alert_id", StringType()),
    StructField("event", StringType()),
    StructField("headline", StringType()),
    StructField("description", StringType()),
    StructField("severity", StringType()),
    StructField("certainty", StringType()),
    StructField("urgency", StringType()),
    StructField("area_desc", StringType()),
    StructField("effective", StringType()),
    StructField("expires", StringType()),
    StructField("ingested_at", StringType()),
])
noaa_alerts_df = spark.createDataFrame(alert_rows, schema=noaa_alerts_schema)
print(f"\nNOAA Weather Alerts DataFrame ({len(alert_rows)} alerts):")
display(noaa_alerts_df)

In [0]:
# Cell 3 — USGS Water Data (NWIS)
# USGS Water Services API is free, no API key required
# Parameters: 00060=discharge(cfs), 00065=gage height(ft), 00010=water temp(C)

usgs_url = (
    "https://waterservices.usgs.gov/nwis/iv/"
    "?format=json&stateCd=nc"
    "&parameterCd=00060,00065,00010"
)

try:
    resp = requests.get(usgs_url, timeout=30)
    resp.raise_for_status()
    usgs_data = resp.json()
    print(f"Fetched USGS water data for NC")
except Exception as e:
    print(f"Failed to fetch USGS data: {e}")
    usgs_data = {"value": {"timeSeries": []}}

# Save raw JSON
usgs_file = os.path.join(USGS_VOLUME, f"usgs_water_{run_ts}.json")
with open(usgs_file, "w") as f:
    json.dump(usgs_data, f, indent=2)
print(f"Saved raw JSON to {usgs_file}")

# Parse USGS JSON time series into rows
time_series = usgs_data.get("value", {}).get("timeSeries", [])
print(f"Number of time series entries: {len(time_series)}")

usgs_rows = []
for ts in time_series:
    site_info = ts.get("sourceInfo", {})
    variable_info = ts.get("variable", {})
    values_list = ts.get("values", [])

    site_codes = site_info.get("siteCode", [])
    site_code = site_codes[0].get("value") if site_codes else None
    site_name = site_info.get("siteName")

    # Get coordinates
    geog = site_info.get("geoLocation", {}).get("geogLocation", {}) if site_info.get("geoLocation") else {}
    latitude = geog.get("latitude")
    longitude = geog.get("longitude")

    var_codes = variable_info.get("variableCode", [])
    variable_code = var_codes[0].get("value") if var_codes else None
    variable_name = variable_info.get("variableName")
    unit_code = variable_info.get("unit", {}).get("unitCode")

    # Get latest value
    for val_entry in values_list:
        for v in val_entry.get("value", []):
            usgs_rows.append({
                "site_code": site_code,
                "site_name": site_name,
                "variable_code": variable_code,
                "variable_name": variable_name,
                "value": float(v.get("value")) if v.get("value") else None,
                "unit": unit_code,
                "latitude": float(latitude) if latitude else None,
                "longitude": float(longitude) if longitude else None,
                "observation_time": v.get("dateTime"),
                "ingested_at": run_ts,
            })

print(f"Parsed {len(usgs_rows)} USGS observation records")

if len(usgs_rows) == 0:
    usgs_rows = [{
        "site_code": None, "site_name": None, "variable_code": None,
        "variable_name": None, "value": None, "unit": None,
        "latitude": None, "longitude": None,
        "observation_time": None, "ingested_at": run_ts,
    }]

usgs_schema = StructType([
    StructField("site_code", StringType()),
    StructField("site_name", StringType()),
    StructField("variable_code", StringType()),
    StructField("variable_name", StringType()),
    StructField("value", DoubleType()),
    StructField("unit", StringType()),
    StructField("latitude", DoubleType()),
    StructField("longitude", DoubleType()),
    StructField("observation_time", StringType()),
    StructField("ingested_at", StringType()),
])

usgs_df = spark.createDataFrame(usgs_rows, schema=usgs_schema)
print("\nUSGS Water Data DataFrame:")
display(usgs_df)

In [0]:
# Cell 3.5 — NCDOT TIMS Crash Data (ArcGIS FeatureServer)
# NCDOT Traffic Incident Management System provides crash & incident data via ArcGIS REST API
# Source: https://services.arcgis.com/NuWFvHYDMVmmxMeM/arcgis/rest/services/NCDOT_TIMSIncidentsHistory/FeatureServer/0
# Free, no API key required. ~50,894 Vehicle Crash records out of 136,819 total incidents.

ncdot_base_url = (
    "https://services.arcgis.com/NuWFvHYDMVmmxMeM"
    "/arcgis/rest/services/NCDOT_TIMSIncidentsHistory/FeatureServer/0"
)

# Fetch crash records (EventType = 'Vehicle Crash'), paginated
# Max 1000 records per request; fetch most recent records for the hackathon
# NOTE: TIMS is a historical archive (most recent records are ~May 2026), not real-time

# WHERE clause: crash events only (no time filter to get all available recent records)
where_clause = "EventType='Vehicle Crash'"

# URL-encode the WHERE clause manually (avoid urllib.parse for simplicity)
import urllib.parse
where_encoded = urllib.parse.quote(where_clause)

all_features = []
offset = 0
page_size = 1000
max_pages = 5  # Limit to 5000 most recent records for performance
has_more = True
pages_fetched = 0

while has_more and pages_fetched < max_pages:
    query_url = (
        f"{ncdot_base_url}/query?f=json"
        f"&where={where_encoded}"
        f"&outFields=*"
        f"&resultRecordCount={page_size}"
        f"&resultOffset={offset}"
        f"&orderByFields=CreationDate+DESC"
    )
    resp = requests.get(query_url, timeout=30)
    resp.raise_for_status()
    page_data = resp.json()
    features = page_data.get("features", [])
    all_features.extend(features)
    print(f"  Fetched page at offset {offset}: {len(features)} records (total: {len(all_features)})")
    if len(features) < page_size:
        has_more = False
    else:
        offset += page_size
    pages_fetched += 1
    time.sleep(0.3)  # Rate limiting

print(f"\nTotal crash records fetched (last 24h): {len(all_features)}")

# Save raw JSON
ncdot_file = os.path.join(NCDOT_VOLUME, f"ncdot_crashes_{run_ts}.json")
raw_data = {"features": all_features}
with open(ncdot_file, "w") as f:
    json.dump(raw_data, f, indent=2)
print(f"Saved raw JSON to {ncdot_file}")

# Parse into rows for Spark DataFrame
crash_rows = []
for feat in all_features:
    props = feat.get("attributes", {})
    geom = feat.get("geometry", {})
    crash_rows.append({
        "object_id": props.get("OBJECTID"),
        "incident_id": props.get("Id"),
        "road": props.get("Road", "").strip() if props.get("Road") else None,
        "division": props.get("Division"),
        "reason": props.get("Reason"),
        "condition": props.get("Condition"),
        "detour": props.get("Detour"),
        "latitude": props.get("Latitude") if props.get("Latitude") else geom.get("y"),
        "longitude": props.get("Longitude") if props.get("Longitude") else geom.get("x"),
        "lanes_affected": props.get("LanesAffected"),
        "event_type": props.get("EventType"),
        "event_subtype": props.get("EventSubType"),
        "start_datetime": datetime.datetime.fromtimestamp(props["StartDateTime"] / 1000).strftime("%Y-%m-%d %H:%M:%S") if props.get("StartDateTime") else None,
        "end_datetime": datetime.datetime.fromtimestamp(props["EndDateTime"] / 1000).strftime("%Y-%m-%d %H:%M:%S") if props.get("EndDateTime") else None,
        "last_update": datetime.datetime.fromtimestamp(props["LastUpdateDateTime"] / 1000).strftime("%Y-%m-%d %H:%M:%S") if props.get("LastUpdateDateTime") else None,
        "is_full_closure": props.get("IsFullClosure"),
        "direction": props.get("Direction"),
        "route_type": props.get("RouteType"),
        "county_name": props.get("CountyName"),
        "location": props.get("Location"),
        "global_id": props.get("GlobalID"),
        "ingested_at": run_ts,
    })

print(f"Parsed {len(crash_rows)} crash records")

if len(crash_rows) == 0:
    crash_rows = [{
        "object_id": None, "incident_id": None, "road": None, "division": None,
        "reason": None, "condition": None, "detour": None,
        "latitude": None, "longitude": None, "lanes_affected": None,
        "event_type": None, "event_subtype": None,
        "start_datetime": None, "end_datetime": None, "last_update": None,
        "is_full_closure": None, "direction": None, "route_type": None,
        "county_name": None, "location": None, "global_id": None,
        "ingested_at": run_ts,
    }]

crash_schema = StructType([
    StructField("object_id", IntegerType()),
    StructField("incident_id", IntegerType()),
    StructField("road", StringType()),
    StructField("division", IntegerType()),
    StructField("reason", StringType()),
    StructField("condition", StringType()),
    StructField("detour", StringType()),
    StructField("latitude", DoubleType()),
    StructField("longitude", DoubleType()),
    StructField("lanes_affected", StringType()),
    StructField("event_type", StringType()),
    StructField("event_subtype", StringType()),
    StructField("start_datetime", StringType()),
    StructField("end_datetime", StringType()),
    StructField("last_update", StringType()),
    StructField("is_full_closure", StringType()),
    StructField("direction", StringType()),
    StructField("route_type", StringType()),
    StructField("county_name", StringType()),
    StructField("location", StringType()),
    StructField("global_id", StringType()),
    StructField("ingested_at", StringType()),
])

crash_df = spark.createDataFrame(crash_rows, schema=crash_schema)
print("\nNCDOT TIMS Crash Data DataFrame:")
display(crash_df)

In [0]:
# Cell 4 — Create Delta Tables
# Determine the current catalog and schema
cat_schema = spark.sql("SELECT current_catalog() as cat, current_schema() as sch").collect()[0]
catalog = cat_schema["cat"]
schema = cat_schema["sch"]
print(f"Current catalog: {catalog}, schema: {schema}")
full_schema = f"{catalog}.{schema}"

# Define table names
tables = {
    "epa_air_quality": epa_df,
    "noaa_weather_observations": noaa_obs_df,
    "noaa_weather_alerts": noaa_alerts_df,
    "usgs_water_data": usgs_df,
    "ncdot_crash_data": crash_df,
}

print("\nCreating Delta tables...")
for table_name, df in tables.items():
    full_table = f"{full_schema}.{table_name}"
    df.write.mode("append").saveAsTable(full_table)
    count = spark.sql(f"SELECT COUNT(*) as cnt FROM {full_table}").collect()[0]["cnt"]
    print(f"  {full_table}: {count} rows")

print("\nAll Delta tables created successfully.")

# NearMe Datastream Ingestion Summary

## Data Sources

| Source | API | Data Collected | Delta Table |
| --- | --- | --- | --- |
| EPA Air Quality | AirNow / Open-Meteo | AQI, PM2.5, PM10, O3, NO2, SO2, CO | `epa_air_quality` |
| NOAA Weather | NWS API | Temperature, Wind, Humidity, Pressure, Alerts | `noaa_weather_observations`, `noaa_weather_alerts` |
| USGS Water | NWIS | Discharge, Gage Height, Water Temperature | `usgs_water_data` |
| NCDOT Crashes | TIMS ArcGIS | Vehicle crashes, road closures, lane impacts | `ncdot_crash_data` |

## Raw JSON Storage

* **EPA**: `/Workspace/Users/ksyogueboule@gmail.com/nearme/volume/EPA/`
* **NOAA**: `/Workspace/Users/ksyogueboule@gmail.com/nearme/volume/NOAA/`
* **USGS**: `/Workspace/Users/ksyogueboule@gmail.com/nearme/volume/USGS/`
* **NCDOT**: `/Workspace/Users/ksyogueboule@gmail.com/nearme/volume/NCDOT/`

## Notes

* All APIs are free and require no API key (except AirNow, which has a public fallback via Open-Meteo)
* Data is scoped to North Carolina (Raleigh area: lat 35.78, lon -78.64)
* Each run saves timestamped raw JSON files for traceability
* Delta tables use append mode — each run adds new records for historical tracking